# B站全站排行榜数据分析

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示 - 使用系统字体
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'STHeiti', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False

print('库导入完成！')

In [ ]:
# 加载数据
video_df = pd.read_csv('B站_Top50视频信息.csv')
comment_df = pd.read_csv('B站_Top50热门评论.csv')

print(f'视频数据: {len(video_df)} 条')
print(f'评论数据: {len(comment_df)} 条')
print('\n视频数据列名:')
print(video_df.columns.tolist())

In [ ]:
# 数据预览
video_df.head(10)

In [ ]:
# 计算互动率
video_df['互动率'] = video_df['点赞数'] / video_df['播放量']
video_df['总互动量'] = video_df['点赞数'] + video_df['弹幕数'] + video_df['评论数']

print('基础统计:')
print(f'平均播放量: {video_df["播放量"].mean():,.0f}')
print(f'平均互动率: {video_df["互动率"].mean():.2%}')
print(f'最高播放量: {video_df["播放量"].max():,}')
print(f'最高互动率: {video_df["互动率"].max():.2%}')

In [ ]:
# 各分区视频数量统计
category_count = video_df['分区'].value_counts()

print('各分区视频数量:')
print(category_count)
print(f'\n共有 {len(category_count)} 个分区')

# 柱状图
fig, ax = plt.subplots(figsize=(12, 8))
x = range(len(category_count))
bars = ax.bar(x, category_count.values, color='skyblue')
ax.set_xticks(x)
ax.set_xticklabels(category_count.index, rotation=45, ha='right')
ax.set_ylabel('视频数量')
ax.set_title('各分区视频数量分布')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# 各分区平均播放量
category_stats = video_df.groupby('分区').agg({
    '播放量': 'mean',
    '点赞数': 'mean',
    '互动率': 'mean'
}).round(2)

category_stats = category_stats.sort_values('播放量', ascending=False)

print('各分区平均播放量:')
print(category_stats)

# 柱状图
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(category_stats.index[::-1], category_stats['播放量'][::-1], color='skyblue')
ax.set_xlabel('平均播放量')
ax.set_title('各分区平均播放量对比')

for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2, 
            f'{width:,.0f}', ha='left', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 播放量与互动率散点图
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(video_df['播放量'], 
                    video_df['互动率'],
                    s=100, alpha=0.7, c='coral', edgecolors='white')

ax.set_xlabel('播放量')
ax.set_ylabel('互动率')
ax.set_title('播放量与互动率关系')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 评论基础统计
print('评论基础统计:')
print(f'总评论数: {len(comment_df)}')
print(f'评论平均点赞数: {comment_df["点赞数"].mean():,.0f}')
print(f'评论最高点赞数: {comment_df["点赞数"].max():,}')

# 评论长度
comment_df['评论长度'] = comment_df['评论内容'].astype(str).apply(len)
print(f'\n评论平均长度: {comment_df["评论长度"].mean():.1f} 字符')

# 评论长度分布
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(comment_df['评论长度'], bins=30, color='lightgreen', edgecolor='white', alpha=0.8)
ax.set_xlabel('评论长度（字符数）')
ax.set_ylabel('评论数量')
ax.set_title('评论长度分布')
ax.axvline(comment_df['评论长度'].mean(), color='red', linestyle='--', 
           label=f'平均值: {comment_df["评论长度"].mean():.0f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 情感分析
from snownlp import SnowNLP

def get_sentiment(text):
    try:
        s = SnowNLP(str(text))
        return s.sentiments
    except:
        return 0.5

comment_df['情感分数'] = comment_df['评论内容'].apply(get_sentiment)
comment_df['情感标签'] = comment_df['情感分数'].apply(
    lambda x: '正面' if x > 0.7 else ('负面' if x < 0.3 else '中性')
)

sentiment_counts = comment_df['情感标签'].value_counts()
print('评论情感分布:')
print(sentiment_counts)
print(f'\n平均情感分数: {comment_df["情感分数"].mean():.3f}')

# 情感分布柱状图
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#FF6B6B', '#95E1D3', '#F8B500']
bars = ax.bar(sentiment_counts.index, sentiment_counts.values, color=colors)
ax.set_ylabel('评论数量')
ax.set_title('评论情感分布')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# 高频词统计
from collections import Counter
import jieba

all_comments = ' '.join(comment_df['评论内容'].astype(str).tolist())
words = jieba.cut(all_comments)
word_list = [word for word in words if len(word) > 1]
word_freq = Counter(word_list)

# 过滤停用词
stopwords = {'的', '了', '是', '在', '我', '有', '和', '就', '不', '人', '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', '你', '会', '着', '没有', '看', '好', '自己', '这'}
word_freq = {k: v for k, v in word_freq.items() if k not in stopwords}

# 高频词TOP20
top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:20]

print('评论高频词TOP20:')
for word, freq in top_words:
    print(f'{word}: {freq}')

# 柱状图
fig, ax = plt.subplots(figsize=(12, 8))
words_list, freqs = zip(*top_words)
bars = ax.barh(list(words_list)[::-1], list(freqs)[::-1], color='orange')
ax.set_xlabel('出现次数')
ax.set_title('评论高频词TOP20')

for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2, 
            f'{int(width)}', ha='left', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# TOP10视频
print('播放量TOP10视频:')
for idx, row in video_df.head(10).iterrows():
    print(f"{row['排名']}. {row['标题'][:30]}... | UP主: {row['UP主']} | 播放量: {row['播放量']:,}")

# TOP10视频柱状图
fig, ax = plt.subplots(figsize=(14, 8))
top10 = video_df.head(10)
short_titles = [t[:10]+'...' for t in top10['标题'][::-1]]
bars = ax.barh(short_titles, top10['播放量'][::-1], color='lightblue')
ax.set_xlabel('播放量')
ax.set_title('播放量TOP10视频')

for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height()/2, 
            f'{width:,.0f}', ha='left', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# 生成分析报告
print('='*50)
print('B站全站排行榜数据分析报告')
print('='*50)

print(f'\n数据概况:')
print(f'- 分析视频: {len(video_df)} 个')
print(f'- 分析评论: {len(comment_df)} 条')
print(f'- 涵盖分区: {len(category_count)} 个')

print(f'\n核心发现:')
print(f'- 热门分区TOP3: {list(category_count.index[:3])}')
print(f'- 平均互动率: {video_df["互动率"].mean():.2%}')
print(f'- 最高播放量: {video_df["播放量"].max():,}')
print(f'- 正面评论占比: {len(comment_df[comment_df["情感标签"]=="正面"])/len(comment_df)*100:.1f}%')

print('\nTOP5视频:')
for idx, row in video_df.head(5).iterrows():
    print(f"{row['排名']}. {row['标题'][:35]}...")

print('\n' + '='*50)
print('分析完成！')
print('='*50)